# Query-Adaptive EfficientRAG: recovery and A–E evaluation
Use a fresh Colab **T4 GPU** runtime. This notebook uses your existing checkpoint-8000 Labeler and checkpoint-500 Filter; it never trains either model. Upload the accompanying `query_adaptive_efficientrag.zip` when prompted. Original Drive artifacts are read-only inputs. Each run uses a new directory.

Historical metrics are recorded evidence, not reproduced results. Start with five questions and inspect `historical_reproduction.json`, `comparison.csv`, and K traces before running all 100. See `AUDIT.md` for the unresolved whole-word reconstruction provenance.

In [ ]:
from google.colab import files, drive
from pathlib import Path
import zipfile, sys, subprocess, json, shutil
uploaded = files.upload()  # Select query_adaptive_efficientrag.zip
archive = next(Path(name) for name in uploaded if name.endswith('.zip'))
ROOT = Path('/content/query_adaptive_efficientrag')
if ROOT.exists():
    raise FileExistsError(f'{ROOT} exists; use the existing folder or a fresh runtime')
with zipfile.ZipFile(archive) as z:
    destination = Path('/content').resolve()
    for name in z.namelist():
        if not (destination / name).resolve().is_relative_to(destination):
            raise ValueError('Unsafe archive path')
    z.extractall(destination)
assert (ROOT / 'run_experiments.py').exists()
drive.mount('/content/drive')

Install only inference requirements. This deliberately avoids the old broad upgrade/force-reinstall cells and training dependencies. After this cell, restart the runtime once via **Runtime → Restart session**, then continue below. Keep Colab's existing PyTorch build. Any pip conflict must be assessed before running experiments; do not assume imports prove compatibility.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements-colab.txt')], check=True)
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=False)

In [ ]:
from pathlib import Path
import sys, subprocess, json, shutil, datetime
ROOT = Path('/content/query_adaptive_efficientrag')
import torch, transformers, tokenizers, faiss, spacy
assert torch.cuda.is_available(), 'Choose a GPU runtime'
assert transformers.__version__ == '4.46.3'
assert tokenizers.__version__ == '0.20.3'
spacy.load('en_core_web_sm')
print('GPU:', torch.cuda.get_device_name(0))
print('Torch / Transformers / FAISS:', torch.__version__, transformers.__version__, faiss.__version__)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=ROOT, check=True)

Locate artifacts by their recorded identities, not by modification time. The search below prefers the exact recorded path, lists fallback candidates, and refuses ambiguous fallbacks. If a required artifact is missing, restore it from your backup before continuing. Do not substitute a different set of 100 questions or corpus.

In [ ]:
DRIVE = Path('/content/drive/MyDrive')
assert DRIVE.exists(), 'Mount Google Drive first'

def locate(preferred, leaf, parent_name=None):
    preferred = Path(preferred)
    if preferred.exists():
        print('Recorded path:', preferred)
        return preferred
    candidates = sorted(p for p in DRIVE.rglob(leaf) if parent_name is None or p.parent.name == parent_name)
    print('Candidates:', *candidates, sep='\n')
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one {leaf}; found {len(candidates)}. Set its path explicitly after inspecting candidates.')
    return candidates[0]

LABELER = locate(DRIVE/'EfficientRAG/EfficientRAG_New_Run/labeler_fixed_full_data/labeler_20260918_031818/checkpoint-8000',
                 'checkpoint-8000', 'labeler_20260918_031818')
FILTER = locate(DRIVE/'EfficientRAG_New_Run/filter_checkpoint_500', 'filter_checkpoint_500')
CORPUS = locate(DRIVE/'EfficientRAG_New_Run/reconstructed_hotpot100/corpus.jsonl', 'corpus.jsonl', 'reconstructed_hotpot100')
EVALUATION = locate(DRIVE/'EfficientRAG_New_Run/reconstructed_hotpot100/valid.jsonl', 'valid.jsonl', 'reconstructed_hotpot100')
for checkpoint in [LABELER, FILTER]:
    assert (checkpoint/'config.json').is_file()
    assert list(checkpoint.glob('*.safetensors')) or list(checkpoint.glob('pytorch_model*.bin'))
    print(checkpoint, json.loads((checkpoint/'config.json').read_text()).get('architectures'))
assert sum(bool(s.strip()) for s in CORPUS.open()) == 991
assert sum(bool(s.strip()) for s in EVALUATION.open()) == 100
historical_candidates = list(DRIVE.rglob('hotpotQA-new-full-labeler8000-filter500-rebuilt.jsonl'))
HISTORICAL = historical_candidates[0] if len(historical_candidates) == 1 else None
print('Historical reference:', HISTORICAL)
if HISTORICAL is None:
    print('Historical trace reproduction cannot be verified until a unique reference is selected.')

Build fresh embeddings for the exact recovered corpus (991 passages; no model training). A new destination prevents stale indexes or mismatched passage text. Downloading Contriever requires network access. The runner hashes the corpus, evaluation data, checkpoint weights, and embeddings; its FAISS index is built in memory from these embeddings.

In [ ]:
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORK = Path('/content') / ('adaptive_run_' + STAMP)
WORK.mkdir(exist_ok=False)
EMBEDDINGS = WORK/'embeddings'
EMBEDDINGS.mkdir()
subprocess.run([sys.executable, 'src/retrievers/passage_embedder.py',
    '--passages', str(CORPUS), '--output_dir', str(EMBEDDINGS),
    '--model_type', 'contriever', '--model_name_or_path', 'facebook/contriever-msmarco',
    '--batch_size', '32'], cwd=ROOT, check=True)
BASE = [sys.executable, 'run_experiments.py', '--labeler', str(LABELER), '--filter', str(FILTER),
        '--corpus', str(CORPUS), '--evaluation', str(EVALUATION), '--embeddings', str(EMBEDDINGS),
        '--max-rounds', '4', '--policy-config', str(ROOT/'policy.json')]
if HISTORICAL is not None:
    BASE += ['--historical-reference', str(HISTORICAL)]
subprocess.run(BASE + ['--limit','5','--verify-baseline','--output',str(WORK/'smoke')], cwd=ROOT, check=True)
print((WORK/'smoke/comparison.csv').read_text())
if (WORK/'smoke/historical_reproduction.json').exists():
    print((WORK/'smoke/historical_reproduction.json').read_text())

Review smoke results. `baseline_verification` must pass for fixed K=5 and 10. If historical traces differ, this is a fresh controlled baseline, not a reproduction of 197/200 and 186/200. Investigate source version (especially the earlier whole-word patch), exact corpus, model revisions, and checkpoint hashes before making a reproduction claim.

The next cell runs the full 100-question A–E comparison. All modes share four maximum rounds, K≤10, and thus a common upper ceiling of 40 passages/question. Fixed K=5 has a stricter realized ceiling of 20; cost is an outcome, not held constant. No oracle information is passed to the controller.

In [ ]:
subprocess.run(BASE + ['--limit','100','--output',str(WORK/'full100')], cwd=ROOT, check=True)
print((WORK/'full100/comparison.csv').read_text())
BACKUP = DRIVE/'EfficientRAG_New_Run'/'dynamic_topk_runs'/STAMP
shutil.copytree(WORK, BACKUP)  # Refuses an existing destination; original artifacts untouched.
print('Saved:', BACKUP)

Inspect `D_dynamic.metrics.json` and `E_full.metrics.json`: `within_question_k_changes` must be nonzero to establish actual per-hop adaptation on these questions. Changes in the global K histogram alone can arise from initial complexity selection. Answer EM/F1 and token usage are intentionally absent: this is a retrieval-only experiment. Repeat timing runs in varied A–E order before making latency claims; one ordered run is exploratory.